# 層和區塊
:label:`sec_model_construction`

之前首次介紹神經網絡時，我們關注的是具有單一輸出的線性模型。
在這裡，整個模型只有一個輸出。
注意，單個神經網絡
（1）接受一些輸入；
（2）生成相應的標量輸出；
（3）具有一組相關 *參數*（parameters），更新這些參數可以優化某目標函數。

然後，當考慮具有多個輸出的網絡時，
我們利用矢量化算法來描述整層神經元。
像單個神經元一樣，層（1）接受一組輸入，
（2）生成相應的輸出，
（3）由一組可調整參數描述。
當我們使用softmax回歸時，一個單層本身就是模型。
然而，即使我們隨後引入了多層感知機，我們仍然可以認為該模型保留了上面所說的基本架構。

對於多層感知機而言，整個模型及其組成層都是這種架構。
整個模型接受原始輸入（特徵），生成輸出（預測），
並包含一些參數（所有組成層的參數集合）。
同樣，每個單獨的層接收輸入（由前一層提供），
生成輸出（到下一層的輸入），並且具有一組可調參數，
這些參數根據從下一層反向傳播的信號進行更新。

事實證明，研究討論“比單個層大”但“比整個模型小”的組件更有價值。
例如，在計算機視覺中廣泛流行的ResNet-152架構就有數百層，
這些層是由*層組*（groups of layers）的反覆模式組成。
這個ResNet架構贏得了2015年ImageNet和COCO計算機視覺比賽
的識別和檢測任務 :cite:`He.Zhang.Ren.ea.2016`。
目前ResNet架構仍然是許多視覺任務的首選架構。
在其他的領域，如自然語言處理和語音，
層組以各種反覆模式排列的類似架構現在也是普遍存在。

為了實現這些複雜的網絡，我們引入了神經網絡*塊*的概念。
*塊*（block）可以描述單個層、由多個層組成的組件或整個模型本身。
使用塊進行抽象的一個好處是可以將一些塊組合成更大的組件，
這一過程通常是遞歸的，如 :numref:`fig_blocks`所示。
通過定義代碼來按需生成任意複雜度的塊，
我們可以通過簡潔的代碼實現複雜的神經網絡。

![多個層被組合成塊，形成更大的模型](../img/blocks.svg)
:label:`fig_blocks`

從編程的角度來看，塊由*類*（class）表示。
它的任何子類必須定義一個將輸入轉換為輸出的前向傳播函數，
並且必須存儲任何必需的參數。
注意，有些塊不需要任何參數。
最後，為了計算梯度，塊必須具有反向傳播函數。
在定義我們自己的塊時，由於自動微分（在 :numref:`sec_autograd` 中引入）
提供了一些後端實現，我們只需要考慮前向傳播函數和必需的參數。

在構造自定義塊之前，(**我們先回顧一下多層感知機**)
（ :numref:`sec_mlp_concise` ）的代碼。
下面的代碼生成一個網絡，其中包含一個具有256個單元和ReLU激活函數的全連接隱藏層，
然後是一個具有10個隱藏單元且不帶激活函數的全連接輸出層。


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


tensor([[ 0.1745, -0.0106,  0.1842,  0.1776,  0.2426,  0.1924, -0.2127,  0.0463,
          0.0653, -0.2894],
        [ 0.2640, -0.1551,  0.0867,  0.1181,  0.2394,  0.1569, -0.1675, -0.0556,
          0.0346, -0.2177]], grad_fn=<AddmmBackward0>)

在這個例子中，我們通過實例化`nn.Sequential`來構建我們的模型，
層的執行順序是作為參數傳遞的。
簡而言之，(**`nn.Sequential`定義了一種特殊的`Module`**)，
即在PyTorch中表示一個塊的類，
它維護了一個由`Module`組成的有序列表。
注意，兩個全連接層都是`Linear`類的實例，
`Linear`類本身就是`Module`的子類。
另外，到目前為止，我們一直在通過`net(X)`調用我們的模型來獲得模型的輸出。
這實際上是`net.__call__(X)`的簡寫。
這個前向傳播函數非常簡單：
它將列表中的每個塊連接在一起，將每個塊的輸出作為下一個塊的輸入。


## [**自定義塊**]

要想直觀地了解塊是如何工作的，最簡單的方法就是自己實現一個。
在實現我們自定義塊之前，我們簡要總結一下每個塊必須提供的基本功能。


1. 將輸入數據作為其前向傳播函數的參數。
1. 通過前向傳播函數來生成輸出。請注意，輸出的形狀可能與輸入的形狀不同。例如，我們上面模型中的第一個全連接的層接收一個20維的輸入，但是返回一個維度為256的輸出。
1. 計算其輸出關於輸入的梯度，可通過其反向傳播函數進行訪問。通常這是自動發生的。
1. 存儲和訪問前向傳播計算所需的參數。
1. 根據需要初始化模型參數。


在下面的代碼片段中，我們從零開始編寫一個塊。
它包含一個多層感知機，其具有256個隱藏單元的隱藏層和一個10維輸出層。
注意，下面的`MLP`類繼承了表示塊的類。
我們的實現只需要提供我們自己的構造函數（Python中的`__init__`函數）和前向傳播函數。


In [2]:
class MLP(nn.Module):
    # 用模型參數聲明層。這裡，我們聲明兩個全連接的層
    def __init__(self):
        # 調用MLP的父類Module的構造函數來執行必要的初始化。
        # 這樣，在類實例化時也可以指定其他函數參數，例如模型參數params（稍後將介紹）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隱藏層
        self.out = nn.Linear(256, 10)  # 輸出層

    # 定義模型的前向傳播，即如何根據輸入X返回所需的模型輸出
    def forward(self, X):
        # 注意，這裡我們使用ReLU的函數版本，其在nn.functional模塊中定義。
        return self.out(F.relu(self.hidden(X)))

我們首先看一下前向傳播函數，它以`X`作為輸入，
計算帶有激活函數的隱藏表示，並輸出其未規範化的輸出值。
在這個`MLP`實現中，兩個層都是實例變量。
要了解這為什麼是合理的，可以想象實例化兩個多層感知機（`net1`和`net2`），
並根據不同的數據對它們進行訓練。
當然，我們希望它們學到兩種不同的模型。

接着我們[**實例化多層感知機的層，然後在每次調用前向傳播函數時調用這些層**]。
注意一些關鍵細節：
首先，我們定制的`__init__`函數通過`super().__init__()`
調用父類的`__init__`函數，
省去了重複編寫模版代碼的痛苦。
然後，我們實例化兩個全連接層，
分別為`self.hidden`和`self.out`。
注意，除非我們實現一個新的運算符，
否則我們不必擔心反向傳播函數或參數初始化，
系統將自動生成這些。

我們來試一下這個函數：


In [3]:
net = MLP()
net(X)

tensor([[ 0.1505, -0.1346, -0.0447,  0.0576,  0.0067,  0.1990,  0.0379, -0.1487,
          0.0015,  0.1753],
        [ 0.1352, -0.0858,  0.0623,  0.0166,  0.0380,  0.2318,  0.0130, -0.0364,
          0.0488,  0.1730]], grad_fn=<AddmmBackward0>)

塊的一個主要優點是它的多功能性。
我們可以子類化塊以創建層（如全連接層的類）、
整個模型（如上面的`MLP`類）或具有中等複雜度的各種組件。
我們在接下來的章節中充分利用了這種多功能性，
比如在處理卷積神經網絡時。

## [**順序塊**]

現在我們可以更仔細地看看`Sequential`類是如何工作的，
回想一下`Sequential`的設計是為了把其他模塊串起來。
為了構建我們自己的簡化的`MySequential`，
我們只需要定義兩個關鍵函數：

1. 一種將塊逐個追加到列表中的函數；
1. 一種前向傳播函數，用於將輸入按追加塊的順序傳遞給塊組成的“鏈條”。

下面的`MySequential`類提供了與默認`Sequential`類相同的功能。


In [4]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 這裡，module是Module子類的一個實例。我們把它保存在'Module'類的成員
            # 變量_modules中。_module的類型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

`__init__`函數將每個模塊逐個添加到有序字典`_modules`中。
讀者可能會好奇為什麼每個`Module`都有一個`_modules`屬性？
以及為什麼我們使用它而不是自己定義一個Python列表？
簡而言之，`_modules`的主要優點是：
在模塊的參數初始化過程中，
系統知道在`_modules`字典中查找需要初始化參數的子塊。


当`MySequential`的前向传播函数被调用时，
每个添加的块都按照它们被添加的顺序执行。
现在可以使用我们的`MySequential`类重新实现多层感知机。


In [5]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[-3.5871e-02,  3.5535e-01, -2.8782e-01,  8.9512e-02,  6.6800e-02,
          1.4851e-02, -2.1741e-01,  6.9478e-02,  1.3193e-02,  1.6730e-01],
        [ 1.5124e-04,  3.0683e-01, -1.3370e-01, -5.5505e-02, -3.9490e-02,
         -4.7512e-03, -2.0483e-01,  3.2258e-02,  1.0951e-02,  8.1884e-02]],
       grad_fn=<AddmmBackward0>)

請注意，`MySequential`的用法與之前為`Sequential`類編寫的代碼相同
（如 :numref:`sec_mlp_concise` 中所述）。

## [**在前向傳播函數中執行代碼**]

`Sequential`類使模型構造變得簡單，
允許我們組合新的架構，而不必定義自己的類。
然而，並不是所有的架構都是簡單的順序架構。
當需要更強的靈活性時，我們需要定義自己的塊。
例如，我們可能希望在前向傳播函數中執行Python的控制流。
此外，我們可能希望執行任意的數學運算，
而不是簡單地依賴預定義的神經網絡層。

到目前為止，
我們網絡中的所有操作都對網絡的激活值及網絡的參數起作用。
然而，有時我們可能希望合并既不是上一層的結果也不是可更新參數的項，
我們稱之為*常數參數*（constant parameter）。
例如，我們需要一個計算函數
$f(\mathbf{x},\mathbf{w}) = c \cdot \mathbf{w}^\top \mathbf{x}$的層，
其中$\mathbf{x}$是輸入，
$\mathbf{w}$是參數，
$c$是某個在優化過程中沒有更新的指定常量。
因此我們實現了一個`FixedHiddenMLP`類，如下所示：


In [6]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不計算梯度的隨機權重參數。因此其在訓練期間保持不變
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用創建的常量參數以及relu和mm函數
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        # 復用全連接層。這相當於兩個全連接層共享參數
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

在這個`FixedHiddenMLP`模型中，我們實現了一個隱藏層，
其權重（`self.rand_weight`）在實例化時被隨機初始化，之後為常量。
這個權重不是一個模型參數，因此它永遠不會被反向傳播更新。
然後，神經網絡將這個固定層的輸出通過一個全連接層。

注意，在返回輸出之前，模型做了一些不尋常的事情：
它運行了一個while循環，在$L_1$範數大於$1$的條件下，
將輸出向量除以$2$，直到它滿足條件為止。
最後，模型返回了`X`中所有項的和。
注意，此操作可能不會常用於在任何實際任務中，
我們只展示如何將任意代碼集成到神經網絡計算的流程中。


In [7]:
net = FixedHiddenMLP()
net(X)

tensor(0.3032, grad_fn=<SumBackward0>)

我們可以[**混合搭配各種組合塊的方法**]。
在下面的例子中，我們以一些想到的方法嵌套塊。


In [8]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.1182, grad_fn=<SumBackward0>)

## 效率


讀者可能會開始擔心操作效率的問題。
畢竟，我們在一個高性能的深度學習庫中進行了大量的字典查找、
代碼執行和許多的其他Python代碼。
Python的問題[全局解釋器鎖](https://wiki.python.org/moin/GlobalInterpreterLock)
是眾所周知的。
在深度學習環境中，我們擔心速度極快的GPU可能要等到CPU運行Python代碼後才能運行另一個作業。


## [**小结**]

* 一個塊可以由許多層組成；一個塊可以由許多塊組成。
* 塊可以包含代碼。
* 塊負責大量的內部處理，包括參數初始化和反向傳播。
* 層和塊的順序連接由`Sequential`塊處理。

## [**练习**]

1. 如果將`MySequential`中儲存區塊的方式更改為Python列表，會出現什麼樣的問題？
1. 實現一個區塊，它以兩個區塊為參數，例如`net1`和`net2`，並返回前向傳播中兩個網路的串聯輸出。這也被稱為平行區塊。
1. 假設我們想要連接同一網路的多個實例。實現一個函數，該函數生成同一個區塊的多個實例，並在此基礎上構建更大的網路。


[Discussions](https://discuss.d2l.ai/t/1827)


練習一:

1. 如果將`MySequential`中儲存區塊的方式更改為Python列表，會出現什麼樣的問題？

我的回答:



如果將`MySequential`中儲存區塊的方式從`_modules` OrderedDict 改為 Python 列表，會出現以下問題：

1. 參數管理問題：
- PyTorch 無法自動追蹤和管理模型參數
- `parameters()` 方法無法正確返回所有子模塊的參數
- 模型的參數無法被正確地注冊到優化器中

2. 設備管理問題：
- `to(device)` 等方法無法正確將模型移動到 GPU/CPU
- 子模塊的設備轉換會失效

3. 保存/加載問題：
- 模型的保存和加載功能會失效
- `state_dict()` 無法正確保存模型狀態

這就是為什麼 PyTorch 使用 `_modules` OrderedDict 而不是普通 Python 列表的原因。它提供了完整的模型參數管理機制。



練習二:

1. 實現一個區塊，它以兩個區塊為參數，例如`net1`和`net2`，並返回前向傳播中兩個網路的串聯輸出。這也被稱為平行區塊。

我的回答:



以下是實現一個平行區塊的代碼：

```python
class ParallelBlock(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2
        
    def forward(self, x):
        # 將兩個網路的輸出在特徵維度上連接
        return torch.cat((self.net1(x), self.net2(x)), dim=1)

# 測試代碼
# 創建兩個簡單的網路
net1 = nn.Sequential(nn.Linear(20, 256), nn.ReLU())
net2 = nn.Sequential(nn.Linear(20, 128), nn.ReLU())

# 創建平行區塊
parallel_net = ParallelBlock(net1, net2)

# 測試
X = torch.randn(2, 20)
output = parallel_net(X)
print(output.shape)  # torch.Size([2, 384]) (256 + 128 = 384)
```

這個實現：
1. 接收兩個網路作為參數
2. 在前向傳播時並行處理輸入
3. 使用 `torch.cat` 將兩個輸出連接在一起
4. 保持了 PyTorch 的自動求導功能

這種結構在需要不同特徵提取路徑的網路中很常見，比如：
- 多尺度特徵提取
- 多模態融合
- 注意力機制



練習三:

1. 假設我們想要連接同一網路的多個實例。實現一個函數，該函數生成同一個區塊的多個實例，並在此基礎上構建更大的網路。

我的回答:



以下是一個實現，可以創建同一個網路的多個實例並將它們連接起來：

````python
class CloneBlock(nn.Module):
    def __init__(self, net, num_clones=3):
        """
        創建指定網路的多個副本並將它們連接起來
        
        參數:
        net: 要克隆的基礎網路
        num_clones: 要創建的克隆數量
        """
        super().__init__()
        # 創建多個相同網路的副本
        self.nets = nn.ModuleList([
            copy.deepcopy(net) for _ in range(num_clones)
        ])
        
    def forward(self, x):
        # 將所有網路的輸出在特徵維度上連接
        return torch.cat([net(x) for net in self.nets], dim=1)

# 測試代碼
# 創建基礎網路
base_net = nn.Sequential(
    nn.Linear(20, 64),
    nn.ReLU()
)

# 創建包含3個克隆的網路
clone_net = CloneBlock(base_net, num_clones=3)

# 測試
X = torch.randn(2, 20)
output = clone_net(X)
print(output.shape)  # torch.Size([2, 192]) (64 * 3 = 192)
````

這個實現的特點：

1. 使用 `copy.deepcopy()` 確保每個網路實例是獨立的
2. 使用 `nn.ModuleList` 正確管理子模塊
3. 每個克隆都有自己的參數集
4. 所有克隆共享相同的架構但有不同的參數

應用場景：
- 集成學習
- 多頭注意力機制
- 特徵增強
- 模型並行化

注意事項：
1. 需要導入 copy 模塊：`import copy`
2. 每個克隆都是獨立訓練的
3. 輸出維度是基礎網路輸出維度的 num_clones 倍
